# 3. Weather data and observability

**Pre-live authoring draft — not ready for participant delivery.**

Run this notebook in a Python kernel started in `beginner/`. Default execution
uses only local teaching assets and Python's standard library, with no network
requests. Live cells are skipped by default. This notebook neither deploys the
weather function nor implements monitoring access or resource cleanup.

Learning goals: distinguish supplied data from live data, understand validation
errors, and distinguish client-observed latency from platform execution metrics.
The deterministic weather function processes supplied data; it does not call an
external provider. Successful offline execution is not platform validation.


## 1. Locate helpers and load the fixture — offline

**Purpose:** inspect a reproducible teaching input without an API key or service.
**Location:** this notebook's Python kernel, started in `beginner/`.
**Expected result:** Amsterdam, 12.5 degrees Celsius, Cloudy.
**Recovery:** restart in `beginner/` if the module path check fails; do not install
packages or fetch replacement data for this exercise.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import sys

MODULE = Path.cwd().resolve().parent
if not (MODULE / "config/lab.example.json").is_file():
    raise RuntimeError("Open this notebook from the module's beginner directory.")
sys.path.insert(0, str(MODULE / "helpers"))
from lab_config import load_config
from invoke import invoke
from traffic import run_traffic

EXAMPLE_CONFIG = MODULE / "config/lab.example.json"
example = load_config(EXAMPLE_CONFIG, require_live=False)
weather = json.loads((MODULE / "fixtures/weather-sample.json").read_text(encoding="utf-8"))
assert weather == {"city": "Amsterdam", "temperature_c": 12.5, "condition": "Cloudy"}
print(json.dumps(weather, indent=2, ensure_ascii=False))
print("Teaching fixture, not a current weather report.")


## 2. Read the function contract — offline

The handler accepts POST with an object containing a non-empty city string,
a finite numeric temperature and a non-empty condition string. City and condition
are trimmed and bounded using JavaScript string length. Temperature may be
negative; a string such as `"12.5"` is not a number in this contract.

**Checkpoint:** locate each validation branch. The code below prints the real
source and constructs the expected fixture response; it does not execute or
independently validate the JavaScript handler.


In [ ]:
print((MODULE / "examples/weather/handler.js").read_text(encoding="utf-8"))
expected_weather = {
    "city": "Amsterdam",
    "temperature_c": 12.5,
    "condition": "Cloudy",
    "message": "Weather in Amsterdam: 12.5°C, Cloudy.",
}
print("Expected handler response:", json.dumps(expected_weather, ensure_ascii=False))


### 2.1 Run the weather tests in your own working copy

**Purpose:** check the JavaScript handler locally before considering a deployed
function. These tests simulate the handler context; they do not test HTTP,
the watchdog, TLS, deployment or an external weather provider.
**Location:** a JupyterLab Bash terminal on the notebook host. Navigate to
the module directory `05-openfaas/`, containing `examples/` and `config/`.
**Prerequisite:** organiser-supplied Node.js 22 and npm, as in Notebook 2.
Do not install packages, use sudo or request a container socket here.

From the module directory, run:

```bash
(
  set -euo pipefail
  test -f config/lab.example.json
  test -f fixtures/weather-sample.json
  mkdir -p workspace
  if [ -e workspace/weather ] || [ -L workspace/weather ]; then
    printf 'STOP: workspace/weather already exists; preserve your edits.\n' >&2
    exit 1
  fi
  cp -R -- examples/weather workspace/weather
  printf 'Created workspace/weather; reference files are unchanged.\n'
)
```

If the directory already exists, inspect it and continue with that copy;
do not replace it. If copying failed, ask for help inspecting partial files.
The module's Git ignore rule excludes `workspace/`, not the reference files.

Then run this separate command, still from the module directory:

```bash
(
  set -euo pipefail
  test -f config/lab.example.json
  cd workspace/weather
  node --version
  npm --version
  node -e 'if (process.versions.node.split(".")[0] !== "22") { console.error("STOP: Node.js 22 is required."); process.exit(1); }'
  npm test
)
```

**Expected baseline:** all 33 tests pass. No `npm install` is needed because
the example uses built-in Node.js modules only. The test file loads
`../../fixtures/weather-sample.json`; this path works because
`workspace/weather/` has the same directory depth as `examples/weather/`.
Do not move the copy to an arbitrary directory without considering that path.
**Recovery:** missing tools require organiser assistance. Resolve any baseline
failures before adding a test; do not change the shared fixture to hide them.

### 2.2 Predict validation outcomes, then add a test

Read the existing tests and predict these outcomes before running anything:

| Change to the fixture input | Expected status | Expected behaviour |
|---|---|---|
| Set `temperature_c` to numeric `-8` | 200 | Preserve the negative temperature |
| Set `temperature_c` to string `"12.5"` | 400 | Return `invalid_temperature` |
| Set `city` to `"  Amsterdam  "` | 200 | Trim city without mutating input |
| Remove `condition` | 400 | Return `invalid_condition` |

Locate the corresponding tests in **your copy** of `handler.test.js`.
`NaN` and infinities are direct JavaScript test inputs, not valid JSON numbers.
Their rejection tests do not establish how an HTTP wrapper parses invalid JSON.

Now open `workspace/weather/handler.test.js` in JupyterLab and append this
complete test. Do not edit the reference handler or shared fixture:

```javascript
test("participant: cold clear weather has a complete response", async () => {
    const result = await invoke({
        city: "  Nairobi  ",
        temperature_c: -3,
        condition: "  Clear  "
    });
    assert.equal(result.status, 200);
    assert.deepEqual(result.body, {
        city: "Nairobi",
        temperature_c: -3,
        condition: "Clear",
        message: "Weather in Nairobi: -3°C, Clear."
    });
});
```

Save, then rerun the test command from section 2.1.
**Expected result:** all 34 tests pass. This is invented test data, not a
claim about actual Nairobi weather. No handler change should be needed.
**Checkpoint:** explain which assertions cover trimming, temperature
preservation and response formatting. Why is this test reproducible?
**Recovery:** inspect the named failure or syntax error. Keep the assertions;
do not weaken the contract to make a test pass.
Your working copy remains available after the terminal commands finish.


## 3. Interpret observations — offline synthetic example

**Purpose:** practise summarising client observations without running traffic.
**Expected result:** three synthetic responses, two successes and one HTTP error.
These numbers are invented teaching inputs, not measurements or validation evidence.

An HTTP response with status 400 is different from a transport failure where no
usable HTTP response is obtained. Client latency includes networking and other
request-path overhead; it is not the handler's execution time. These samples
cannot establish throughput, scaling, platform error rates or resource usage.


In [ ]:
synthetic_statuses = [200, 400, 200]
synthetic_latencies = [0.25, 0.50, 0.75]
synthetic_summary = {
    "completed_requests": len(synthetic_statuses),
    "status_counts": dict(Counter(synthetic_statuses)),
    "client_latency_seconds": {
        "minimum": min(synthetic_latencies),
        "mean": sum(synthetic_latencies) / len(synthetic_latencies),
        "maximum": max(synthetic_latencies),
    },
}
assert synthetic_summary["status_counts"] == {200: 2, 400: 1}
print("SYNTHETIC TEACHING EXAMPLE — NOT MEASURED")
print(json.dumps(synthetic_summary, indent=2))


### 3.1 Interpret the synthetic observations

**Action:** use the printed summary above to answer these questions before
reading the checkpoint answers. Do not enable live traffic for this exercise.

1. How many usable HTTP responses were recorded, and how many had status 400?
2. What fraction of these synthetic responses were successful status 200?
3. What are the minimum, mean and maximum client latencies in milliseconds?
4. Does status 400 show a transport failure or prove which component rejected
   the request?
5. Can these values tell you handler duration, throughput or autoscaling?

**Checkpoint answers:** three HTTP responses; one status 400; two of three
responses (approximately 66.7%) have status 200. The minimum, mean and maximum
latencies are 250, 500 and 750 milliseconds. Status 400 is an HTTP error
response, not a transport failure, and the status alone does not identify its
source. These synthetic values establish no actual system performance.

The summary uses the same field names as the bounded-traffic helper, but its
inputs are invented. When printed as JSON, integer status keys appear as
strings such as `"200"`; this does not change the underlying HTTP status.
The helper measures each client request, excluding its configured pause
between requests. The reciprocal of mean client latency is not a measured
throughput figure for the batch.

**Recovery:** rerun only the synthetic cell if you changed its values. Keep
the synthetic label and explain any changed counts; do not present the output
as measurements from a deployed function.


## 4. Live execution gate

**STOP until the organiser has validated the assigned lab, deployed owned weather
function, gateway route, TLS and invocation access.** Notebook 2 does not yet
implement that deployment path. Do not bypass this dependency.

The helper currently has no credential support. If authentication is required,
stop until the approved mechanism is implemented and tested. Do not disable TLS
verification, include credentials in configuration, or borrow another lab's route.
A valid configuration is not evidence of readiness or isolation.

Keep `RUN_LIVE = False` for offline review. After the full execution contract is
validated, an explicit change to `True` enables the invocation cells. Bounded
traffic also requires the separate `RUN_TRAFFIC` switch in section 6.


In [ ]:
RUN_LIVE = False
LOCAL_CONFIG = MODULE / "config/lab.local.json"
if RUN_LIVE:
    live_config = load_config(LOCAL_CONFIG)
    print("Configuration accepted; readiness must already have been validated.")
else:
    print("SKIPPED: live configuration and execution.")


## 5. Invoke deterministic weather — gated live actions

**Purpose:** verify supplied data and predictable validation errors against the
actual assigned function. **Location:** the assigned lab's Python kernel.
**Expected candidate results:** fixture response 200, followed by a string
 temperature rejected with 400 and `invalid_temperature`. These are two requests.

**Recovery:** stop on an unexpected status, non-JSON response, access or transport
failure. Report the mismatch to the organiser without printing credentials or
raw parser error bodies. Wrapper-only observations do not guarantee gateway
behaviour. Malformed JSON can be rejected before handler execution with HTML.


In [ ]:
if RUN_LIVE:
    result = invoke(LOCAL_CONFIG, "weather", weather)
    print("Fixture response status:", result.status)
    assert result.status == 200
    assert result.json() == expected_weather
    print("Fixture response:", result.json())

    invalid_weather = dict(weather, temperature_c="12.5")
    invalid_result = invoke(LOCAL_CONFIG, "weather", invalid_weather)
    print("Invalid temperature status:", invalid_result.status)
    assert invalid_result.status == 400
    assert invalid_result.json() == {"error": "invalid_temperature"}
else:
    print("SKIPPED: weather invocation and validation requests.")


## 6. Generate bounded sequential traffic — separately gated live action

**Purpose:** collect client observations for later comparison with validated
monitoring. The helper uses the organiser-supplied request count and pause, with
one request at a time. It pauses after a response, not on a fixed-rate schedule.
There are no automatic retries or background jobs. Transport failures stop the
exercise; HTTP error statuses are counted as responses.

Review and obtain approval for the displayed settings before changing
`RUN_TRAFFIC` to `True`. Do not increase traffic to try to force autoscaling.
The request timeout is a socket timeout, not a strict total wall-clock deadline.
A failed run raises an error and does not return a partial aggregate report.
Rerunning this cell sends a new batch; requests from section 5 are additional.


In [ ]:
RUN_TRAFFIC = False
if RUN_LIVE:
    print("Configured traffic limits:", live_config["load"])
if RUN_TRAFFIC and not RUN_LIVE:
    raise RuntimeError("Traffic requires validated live execution first.")
if RUN_LIVE and RUN_TRAFFIC:
    traffic_summary = run_traffic(LOCAL_CONFIG, "weather", weather)
    print("Measured client observations:")
    print(json.dumps(traffic_summary, indent=2))
else:
    print("SKIPPED: bounded live traffic.")


## 7. Platform monitoring — BLOCKED in this draft

No monitoring query or browser action is implemented here. The organiser must
validate authorised routes, metric names, function labels and time windows on
the selected platform before a runnable exercise can be added.

The future exercise must compare client observations with verified invocation,
error and duration metrics for the assigned function, accounting for scrape
intervals and aggregation. A missing series does not prove zero requests.
Prometheus counters accumulate; use a validated time-window interpretation.
Do not infer handler time from client latency, compare unrelated scopes, or
claim scaling behaviour from this small sequential exercise.

## 8. Optional external weather extension — BLOCKED in this draft

The core fixture requires no external service. A future extension needs a
selected provider, protected credential handling where needed, validated outbound
access, bounded timeouts and explicit handling of HTTP errors, unavailable
services and invalid responses. Provider data must be mapped to this handler's
input schema. Never put API keys in notebooks, fixtures, images or printed URLs.
No provider request or external-failure implementation is included here.

## 9. Scoped cleanup — BLOCKED in this draft

No deletion command is provided until assigned-resource permissions and negative
cross-group tests pass. The future participant action must remove only owned
exercise functions, verify removal, and preserve the organiser-prepared greeting.
Platform teardown, registry retention and infrastructure cleanup remain organiser
responsibilities. A prefix is not sufficient authorisation for deletion.

## 10. Review

- Why is a supplied fixture useful for reproducibility?
- Why does a string temperature fail even when it looks numeric?
- How do an HTTP validation error and a transport failure differ?
- What additional evidence is needed to interpret platform metrics?

This draft explores data and client-observation concepts, but does not provide
complete live monitoring, external-service failure handling or cleanup coverage.
All three notebooks require a fresh validated lab rehearsal before delivery.
